# Constant Acceleration MPC Testing

In [ ]:
%matplotlib ipympl

import jax
jax.config.update("jax_enable_x64", True)
# jax.config.update("jax_compilation_cache_dir", "/tmp/jax_cache")
# jax.config.update("jax_persistent_cache_min_entry_size_bytes", -1)
# jax.config.update("jax_persistent_cache_min_compile_time_secs", 0)
# jax.config.update('jax_disable_jit', True)

In [ ]:
import functools
import jax.numpy as jnp
import numpy as np
import tqdm

import matplotlib.pyplot as plt

import exp_mpc.stewart_min.const as const
import exp_mpc.stewart_min.vest as vest
import exp_mpc.stewart_min.quartic_cost as quartic_cost
import exp_mpc.stewart_min.opt as opt
import exp_mpc.stewart_min.viz as viz
import exp_mpc.stewart_min.mp_mpl as mp_mpl

In [ ]:
# general setup

T = 20.0  # s
num_steps = int(T / const.dt)
n = 200  # horizon

acc_ref = jnp.array([1.0, 0.0, 0.0]) + const.gravity
acc_ref = jnp.tile(A=acc_ref, reps=(n, 1))
omega_ref = jnp.array([0.0, 0.0, 0.1])
omega_ref = jnp.tile(A=omega_ref, reps=(n, 1))

gravity_ref = jnp.tile(A=const.gravity, reps=(n, 1))
zero_ref = jnp.zeros((n, 3))

In [ ]:
# cost setup

weights = opt.ExpWeights(
    acc=jnp.array([1e1, 1e1, 1e0]),
    omega=jnp.array([1e1, 1e1, 1e1]),
    alpha_acc=jnp.array([0.0]),
    alpha_omega=jnp.array([0.0]),
)

margins = [0.2, 0.1]
sizes = [1.0, 2**3, 2**8]
euler_margins = [0.2 / 3.0, 0.1 / 3.0]
euler_sizes = [2**0, 2**3, 2**8]

leg_cost = quartic_cost.QuarticCost.from_bounds(
    margins=[0.3, 0.2, 0.1],
    sizes=[1.0, 2**3, 2**5, 2**10],
    low=const.leg_min,
    high=const.leg_max,
    center=const.lengths_home,
)
leg_vel_cost = quartic_cost.QuarticCost.from_bounds(
    margins=margins,
    sizes = sizes,
    low=-const.max_leg_vel,
    high=const.max_leg_vel,
)
joint_angle_cost = quartic_cost.QuarticCost.from_bounds(
    margins=margins,
    sizes=sizes,
    low=-const.joint_max_angle,
    high=const.joint_max_angle,
)
roll_cost = quartic_cost.QuarticCost.from_bounds(
    margins=euler_margins,
    sizes=euler_sizes,
    low=-const.max_roll,
    high=const.max_roll,
)
pitch_cost = quartic_cost.QuarticCost.from_bounds(
    margins=euler_margins,
    sizes=euler_sizes,
    low=-const.max_pitch,
    high=const.max_pitch,
)
yaw_cost = quartic_cost.QuarticCost.from_bounds(
    margins=euler_margins,
    sizes=euler_sizes,
    low=-const.max_rotary_yaw,
    high=const.max_rotary_yaw,
)
yaw_dot_cost = quartic_cost.QuarticCost.from_bounds(
    margins=euler_margins,
    sizes=euler_sizes,
    low=-const.max_rotary_vel,
    high=const.max_rotary_vel,
)
cost_terms = opt.CostTerms(
    leg_cost=leg_cost,
    leg_vel_cost=leg_vel_cost,
    joint_angle_cost=joint_angle_cost,
    roll_cost=roll_cost,
    pitch_cost=pitch_cost,
    yaw_cost=yaw_cost,
    yaw_dot_cost=yaw_dot_cost,
)

In [ ]:
dt = const.dt
dt_mpc = const.dt * 2.0
tf_acc = vest.spec_refs["acc0"][0]
tf_omega = vest.spec_refs["omega0"][0]
vspec_acc = vest.VSpec.transfer2vspec(tf_acc, dt=dt, earth_moon_v0=True)
vspec_omega = vest.VSpec.transfer2vspec(tf_omega, dt=dt)
vspec_acc_mpc = vest.VSpec.transfer2vspec(tf_acc, dt=dt_mpc, earth_moon_v0=True)
vspec_omega_mpc = vest.VSpec.transfer2vspec(tf_omega, dt=dt_mpc)
train_step = functools.partial(
    opt.train_step_with_cost,
    weights,
    cost_terms,
    dt=dt,
    dt_mpc=dt_mpc,
    opt_options={"maxiter": 3, "maxls": 1},
    vspec_acc=vspec_acc,
    vspec_omega=vspec_omega,
    vspec_acc_mpc=vspec_acc_mpc,
    vspec_omega_mpc=vspec_omega_mpc,
    unroll=False,
    use_terminal=True,
)

In [ ]:
# run setup
train_state = opt.TrainState.zero_init(n, vspec_acc=vspec_acc, vspec_omega=vspec_omega, vstate0_mode=("earth", "earth"))
train_list = []
times = []
sol_list = []
res_list = []

In [ ]:
# run
for i in tqdm.tqdm(range(6 * num_steps)):
    if i <= num_steps:
        aref = acc_ref
        oref = omega_ref
    else:
        aref = gravity_ref
        oref = zero_ref
    train_state, sol, res, t_tot = train_step(aref, oref, train_state)
    train_list.append(train_state)
    sol_list.append(sol)
    res_list.append(res)
    times.append(t_tot)

In [ ]:
freqs = 1.0 / np.array(times)
print(f"{float(np.min(freqs)):.2f}, {float(np.max(freqs)):.2f}, {float(np.mean(freqs)):.2f}, {float(np.std(freqs)):.2f}")

In [ ]:
plt.close("all")

In [ ]:
sol_list_end = []
extra_steps = 0

# sol_list_end = viz.split_tablesol(sol_list[-1])
# extra_steps = sol_list[-1].u.shape[0] - 1

trajectory = sol_list + sol_list_end
references= {
    "xyz-acceleration": jnp.tile(A=acc_ref[0], reps=(len(trajectory), 1)),
    "angular-velocity": jnp.tile(A=omega_ref[0], reps=(len(trajectory), 1)),
}

In [ ]:
mpc_human_fig = viz.plot_human_trajectory(trajectory=trajectory, references=references)

In [ ]:
mpc_vestibular_fig = viz.plot_vestibular_trajectory(trajectory=trajectory)

In [ ]:
mpc_table_fig = viz.plot_cartesian_table_trajectory(trajectory=trajectory)

In [ ]:
mpc_actuator_fig = viz.plot_actuator_trajectory(trajectory=trajectory)

In [ ]:
import pickle
rstate = {
    "trajectory": trajectory,
    "references": references,
    "weights": weights,
    "cost_terms": cost_terms,
}
with open("data/const_acc_400_state.pickle", "wb") as f:
    pickle.dump(rstate, f)

In [ ]:
# plot cost history
def cost_fun(state: opt.TrainState) -> float:
    return float(opt.cost_flat_jax(
        weights=weights,
        cost_terms=cost_terms,
        acc_ref=acc_ref,
        omega_ref=omega_ref,
        rstate0=state.rstate0,
        vstate0_irl=state.vstate0_irl,
        vstate0_sim=state.vstate0_sim,
        control0=state.control0,
        control_flat=state.control_flat,
        dt=dt_mpc,
        vspec_acc=vspec_acc_mpc,
        vspec_omega=vspec_omega_mpc,
        use_rotary=True,
        use_terminal=True,
    ))

costs = [cost_fun(state) for state in train_list]

fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(7, 4))
ax.set_title("Cost history")
ax.set_xlabel("iter")
ax.set_ylabel("cost")
ax.plot(costs)
ax.grid()
fig.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(7, 4))
for i in range(6):
    ax.plot(sol_list[-1].u.control[:, i], label=f"{i}")
ax.grid()
ax.legend()
plt.show()

In [ ]:
assert False

## animations

(WARNING: can take a long time.
Usually about as long as the video being generated.)

In [ ]:
trajectory = sol_list + sol_list_end
references={
    "xyz-acceleration": jnp.tile(
        A=acc_ref[0], reps=(len(sol_list) + len(sol_list_end), 1)
    ),
    "angular-velocity": jnp.tile(
        A=omega_ref[0], reps=(len(sol_list) + len(sol_list_end), 1)
    ),
}

In [ ]:
mp_mpl.call_mp_animate_trajectory(
    file_name="data/const_acc_3d.mp4",
    trajectory=trajectory,
)

In [ ]:
# mp_mpl.call_mp_animate_human_trajectory(
#     file_name="data/const_acc_human.mp4",
#     trajectory=trajectory,
#     references=references,
# )

In [ ]:
# reps = (len(trajectory), 1, 1)
# mp_mpl.call_mp_animate_cost_trajectory(
#     file_name="data/const_acc_cost.mp4",
#     acc_refs=jnp.tile(acc_ref, reps),
#     omega_refs=jnp.tile(omega_ref, reps),
#     weights=weights,
#     cost_terms=cost_terms,
#     trajectory=trajectory,
# )